# Resume Genie: An AI-Powered Career Suite
### Industry-Grade End-to-End Notebook | GROQ LLM | LangChain | MongoDB | MLflow + DagsHub

---

**Four AI-powered tools, one notebook:**

| Module | What it does |
|--------|-------------|
| **Module 1 — Resume Checker** | Standalone resume evaluation: score, strengths, weaknesses, recommended skills, career steps |
| **Module 2 — Resume Scorer / JD Matcher** | Resume vs Job Description deep analysis: keyword match, ATS score, skill gaps |
| **Module 3 — Cover Letter Generator** | Tailored cover letter from resume + JD, streaming output, downloadable |
| **Module 4 — AI Career Coach** | Multi-turn conversational coach grounded in your resume |

> **LLM:** GROQ `llama-3.1-8b-instant` (primary) with OpenAI `gpt-4o-mini` as optional fallback  
> **PDF Parsing:** LangChain `PyPDFLoader`  
> **Tracking:** MLflow / DagsHub  
> **Persistence:** MongoDB  
> **No Streamlit:** Pure Colab-native execution


## 0. Environment Setup — Secrets & Tracking

In [1]:
import os
from google.colab import userdata

# ── MongoDB — stores every run, resume text, and generated outputs ────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub — tracks all LLM calls, scores, and metrics ─────────
USE_DAGSHUB = True  # Set False to use local file-based MLflow

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Fallback: local mlruns/ folder inside Colab session
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

# ── GROQ API — primary LLM provider (instant inference) ───────────────────
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# ── OpenAI — optional fallback LLM ────────────────────────────────────────
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

print('Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")
print(f"   GROQ_API_KEY        = {'SET' if os.environ.get('GROQ_API_KEY') else 'MISSING'}")
print(f"   MONGO_DB_URL        = {'SET' if os.environ.get('MONGO_DB_URL') else 'MISSING'}")

Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow
   GROQ_API_KEY        = SET
   MONGO_DB_URL        = SET


## 1. Install Dependencies

In [2]:
%%capture install_log
# Core LangChain + LLM providers
!pip install langchain langchain_community langchain_core langchain_groq
# OpenAI optional fallback
!pip install langchain_openai
# PDF parsing
!pip install pypdf
# Tracking & persistence
!pip install mlflow dagshub pymongo
# Utilities
!pip install sentence_transformers python-dotenv

In [3]:
# Print install summary as plain stream output (no widget spinners)
tail = install_log.stdout[-1500:] if len(install_log.stdout) > 1500 else install_log.stdout
print(tail or "All packages installed successfully.")
print("Installation complete.")

equirement already satisfied: mpmath<1.4,>=1.1.0 in /usr/local/lib/python3.12/dist-packages (from sympy>=1.13.3->torch>=1.11.0->sentence_transformers) (1.3.0)

Installation complete.


## 2. Imports & Shared Utilities

In [4]:
import os, json, datetime, uuid, time, re, tempfile
from pathlib import Path
from typing import Optional

# ── LangChain core ────────────────────────────────────────────────────────
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader

# ── LLM providers ─────────────────────────────────────────────────────────
from langchain_groq import ChatGroq

# Optional: OpenAI fallback (uncomment if OPENAI_API_KEY is set)
# from langchain_openai import ChatOpenAI

# ── Tracking & persistence ────────────────────────────────────────────────
import mlflow
import dagshub
from pymongo import MongoClient

print("All imports successful.")

All imports successful.


## 3. LLM Factory — GROQ Primary / OpenAI Fallback

In [5]:
# ── LLM Configuration ─────────────────────────────────────────────────────
# GROQ instant model: blazing fast inference, free tier available
# Swap to 'llama-3.3-70b-versatile' for higher quality at slower speed
GROQ_MODEL   = "llama-3.1-8b-instant"
OPENAI_MODEL = "gpt-4o-mini"              # Optional fallback

LLM_PROVIDER = "groq"    # Change to "openai" to use OpenAI instead

def get_llm(temperature: float = 0.2, max_tokens: int = 2000):
    """
    LLM Factory — returns the configured LLM instance.
    Priority: GROQ (primary) > OpenAI (fallback).
    All parameters are overridable per-module.
    """
    if LLM_PROVIDER == "groq":
        groq_key = os.environ.get("GROQ_API_KEY")
        if not groq_key:
            raise ValueError("GROQ_API_KEY not set. Check Colab Secrets.")
        return ChatGroq(
            model=GROQ_MODEL,
            api_key=groq_key,
            temperature=temperature,
            max_tokens=max_tokens,
        )
    elif LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        openai_key = os.environ.get("OPENAI_API_KEY")
        if not openai_key:
            raise ValueError("OPENAI_API_KEY not set. Check Colab Secrets.")
        return ChatOpenAI(
            model=OPENAI_MODEL,
            api_key=openai_key,
            temperature=temperature,
            max_tokens=max_tokens,
        )
    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

# Validate LLM initialises correctly
llm = get_llm()
test_resp = llm.invoke("Reply with exactly: OK")
print(f"LLM provider  : {LLM_PROVIDER.upper()}")
print(f"Model         : {GROQ_MODEL if LLM_PROVIDER == 'groq' else OPENAI_MODEL}")
print(f"Health check  : {test_resp.content.strip()}")

LLM provider  : GROQ
Model         : llama-3.1-8b-instant
Health check  : OK


## 4. MLflow / DagsHub + MongoDB Initialisation

In [6]:
# ── DagsHub MLflow init ───────────────────────────────────────────────────
if USE_DAGSHUB:
    dagshub.init(
        repo_owner=os.environ['MLFLOW_TRACKING_USERNAME'],
        repo_name="resume-genie",
        mlflow=True
    )

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment("resume_genie_suite")
print(f"MLflow URI  : {mlflow.get_tracking_uri()}")
print(f"Experiment  : resume_genie_suite")

# ── MongoDB collections ───────────────────────────────────────────────────
# resume_runs    : one document per tool invocation (metadata + output)
# resume_texts   : cached extracted PDF texts (avoid re-parsing)
# chat_sessions  : full multi-turn career coach conversation logs
mongo_client     = MongoClient(os.environ['MONGO_DB_URL'])
mongo_db         = mongo_client['resume_genie']
col_runs         = mongo_db['resume_runs']
col_resume_texts = mongo_db['resume_texts']
col_chat_sessions= mongo_db['chat_sessions']

try:
    mongo_client.admin.command('ping')
    print("MongoDB connected successfully.")
except Exception as e:
    print(f"MongoDB connection failed: {e}")

# ── Shared logging helper ─────────────────────────────────────────────────
def log_run(tool: str, inputs: dict, output: str, metrics: dict = None,
            run_id: str = None) -> str:
    """
    Persist a tool invocation to MongoDB and log metrics to MLflow.
    Returns the run_id for cross-referencing.
    """
    run_id = run_id or str(uuid.uuid4())[:8]
    ts     = datetime.datetime.utcnow().isoformat()

    # MongoDB document
    col_runs.insert_one({
        "run_id":    run_id,
        "tool":      tool,
        "timestamp": ts,
        "llm":       f"{LLM_PROVIDER}/{GROQ_MODEL if LLM_PROVIDER == 'groq' else OPENAI_MODEL}",
        "inputs":    inputs,
        "output":    output,
        "metrics":   metrics or {}
    })

    # MLflow run
    with mlflow.start_run(run_name=f"{tool}_{run_id}"):
        mlflow.log_param("tool",     tool)
        mlflow.log_param("llm",      LLM_PROVIDER)
        mlflow.log_param("model",    GROQ_MODEL if LLM_PROVIDER == 'groq' else OPENAI_MODEL)
        mlflow.log_text(output,      "output.txt")
        if metrics:
            mlflow.log_metrics(metrics)

    return run_id

print("Logging helpers ready.")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b58a34e9-c48e-4b73-9707-c734e1e4ef1d&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=074f7f86a5581cd522f4f204c5cfefeee605c87395c0a094c8c64656eebd59d8




Accessing as prithusarkar90

Repository resume-genie doesn't exist, creating it under current user.

Initialized MLflow to track repo "prithusarkar90/resume-genie"

Repository prithusarkar90/resume-genie initialized!

2026/04/10 15:23:39 INFO mlflow.tracking.fluent: Experiment with name 'resume_genie_suite' does not exist. Creating a new experiment.


MLflow URI  : https://dagshub.com/prithusarkar90/resume-genie.mlflow
Experiment  : resume_genie_suite
MongoDB connected successfully.
Logging helpers ready.


## 5. PDF Parsing Utilities

In [7]:
def extract_pdf_text(pdf_path: str) -> str:
    """
    Extract all text from a PDF file using LangChain's PyPDFLoader.
    Handles multi-page documents and cleans whitespace artifacts.

    Args:
        pdf_path: Absolute path to the PDF file on disk.
    Returns:
        Concatenated text content from all pages.
    """
    loader    = PyPDFLoader(pdf_path)
    documents = loader.load()
    text      = "\n\n".join(doc.page_content for doc in documents)
    # Remove excessive blank lines introduced by PDF whitespace
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    return text

def upload_and_extract(prompt_msg: str = "Upload your resume PDF:") -> tuple[str, str]:
    """
    Colab-native file upload + PDF extraction helper.
    Returns (extracted_text, filename).
    Caches extracted text in MongoDB to avoid re-parsing on re-runs.
    """
    from google.colab import files

    print(prompt_msg)
    uploaded = files.upload()       # Opens Colab file picker

    if not uploaded:
        raise FileNotFoundError("No file uploaded.")

    filename = list(uploaded.keys())[0]
    pdf_bytes = uploaded[filename]

    # Write bytes to a temp file so PyPDFLoader can read it
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(pdf_bytes)
        tmp_path = tmp.name

    try:
        resume_text = extract_pdf_text(tmp_path)
    finally:
        os.unlink(tmp_path)    # Always clean up temp file

    if not resume_text.strip():
        raise ValueError("No readable text found in the uploaded PDF.")

    # Cache in MongoDB for session reuse
    col_resume_texts.replace_one(
        {"filename": filename},
        {"filename": filename, "text": resume_text,
         "uploaded_at": datetime.datetime.utcnow().isoformat()},
        upsert=True
    )

    print(f"Extracted {len(resume_text)} characters from '{filename}'.")
    print(f"Preview (first 300 chars):\n{resume_text[:300]}...")
    return resume_text, filename

def parse_score_from_output(text: str) -> Optional[float]:
    """
    Attempt to extract a numeric score (X/100) from LLM output.
    Returns None if no score pattern is found.
    """
    match = re.search(r'(?:Score|score)[:\s]+([0-9]+)\s*/\s*100', text)
    if match:
        return float(match.group(1))
    return None

print("PDF utilities ready.")

PDF utilities ready.


## 6. Prompt Templates — All Four Tools

In [9]:
# ── MODULE 1: Resume Checker Prompt ──────────────────────────────────────
# Standalone evaluation — no JD required. Covers clarity, ATS, career path.
RESUME_CHECKER_PROMPT = PromptTemplate(
    input_variables=["context"],
    template="""You are an advanced resume evaluation assistant with expertise in ATS systems,
recruitment best practices, and career coaching across all industries.

Analyze the provided resume text and produce a thorough standalone evaluation.
Base EVERY finding strictly and exclusively on the actual resume content — do not
invent, assume, or add any facts not present in the document.

Your response MUST follow this exact structure (headings as written):

1. Score: X/100
2. Strengths:
   - point one (be specific, cite resume evidence)
   - point two
   - point three (minimum 3 points)
3. Weaknesses / Areas for Improvement:
   - point one (actionable, specific)
   - point two
   - point three (minimum 3 points)
4. Skills Explicitly Mentioned:
   - skill 1
   - skill 2
   (list every technical and soft skill you find)
5. Recommended Additional Skills: (to improve ATS score and future-proof the resume)
   - suggestion 1
   - suggestion 2
6. Suggested Next Career Steps / Roles:
   - realistic next role 1 (based on current experience level)
   - realistic next role 2
   - longer-term direction (2-3 years out)

Be specific, honest, constructive, and professional.

Resume text:
{context}"""
)

# ── MODULE 2: Resume Scorer / JD Matcher Prompt ───────────────────────────
# Deep match analysis between resume and a specific job description.
RESUME_SCORER_PROMPT = PromptTemplate(
    input_variables=["job_description", "context"],
    template="""You are an expert resume scorer and ATS optimization specialist with deep knowledge
of recruitment practices across all industries.

Task: Carefully analyze how well the candidate's resume matches the job description.
Base EVERY statement, score, and suggestion strictly and exclusively on the content
actually present in the provided resume and job description.
Do NOT invent, assume, or add any experience, skills, or facts not in the documents.

Job Description:
{job_description}

Candidate Resume:
{context}

Produce the analysis using EXACTLY the following structure (do not add/remove sections):

Score: [integer]/100
Overall Match: [integer]%

Keywords matched:
- [important keywords/phrases from JD that DO appear in the resume]

Missing keywords:
- [important keywords/phrases from JD completely absent in the resume]

Readability Score: [integer]/100
ATS Compatibility Score: [integer]/100

2-liner summary:
[One strong sentence summarizing overall fit]
[One strong sentence naming the single biggest weakness]

Skill gap analysis:
- [gap 1 → Missing/weak: X → needed for Y part of the role]
- [4-8 bullets max, most impactful gaps only]

Overall improvement suggestions:
- [Prioritized, actionable — strongest bang-for-buck improvements first]
- [Include both content additions and ATS/formatting tips]

Industry specific feedback:
- [2-5 bullets tailored to this role's industry — certifications, metrics, tools, etc.]

Scoring rubrics applied:
- Score: keyword presence + skill relevance + experience recency + quantified achievements + progression
- Match %: estimated probability of passing initial ATS + recruiter screen
- Readability: clarity, grammar, action verbs, formatting, length, absence of fluff
- ATS: standard section headings, keyword density (not stuffing), machine-readable layout

Be honest. If the match is very poor, say so clearly and constructively."""
)

# ── MODULE 3: Cover Letter Generator Prompt ───────────────────────────────
# Produces a ready-to-send professional cover letter (300-450 words).
COVER_LETTER_PROMPT = PromptTemplate(
    input_variables=["job_description", "resume_text"],
    template="""Write a professional, compelling cover letter (300-450 words) tailored specifically
to the job description below.

Emphasize the candidate's most relevant experience, skills, and achievements that
directly match or exceed the job requirements. Use concrete examples from the resume
where possible. Show genuine enthusiasm for the role without fabricating any information.

Structure in standard business format:
- Opening paragraph: state the position + brief why you are a strong fit
- 1-2 body paragraphs: highlight strongest matching qualifications with evidence
- Closing paragraph: reiterate interest, call to action, professional thanks

Job Description:
{job_description}

Candidate Resume:
{resume_text}

IMPORTANT: Do not invent any experience, skills, or facts not present in the resume.
Write in first person. Do not include placeholder text like [Your Name] — write the letter
as a complete, ready-to-send document based on the resume content."""
)

# ── MODULE 4: Career Coach System Prompt ──────────────────────────────────
# Grounds the multi-turn chatbot in the candidate's actual resume.
CAREER_COACH_SYSTEM = """You are an expert career coach, resume mentor, and interview preparation specialist.

You help candidates with:
- Career guidance and progression planning
- Resume review and targeted improvements
- Interview preparation (questions, answers, storytelling)
- Job search strategy (targeting, networking, applications)
- Skill gap analysis and learning recommendations
- Salary negotiation tactics
- Personal brand and LinkedIn profile optimization

IMPORTANT RULES:
1. Ground all advice in the candidate's actual resume provided below.
2. Be specific — cite real experience and skills from the resume in your suggestions.
3. Be honest about weaknesses — growth comes from truthful feedback.
4. Keep responses concise and actionable (max 300 words unless asked for more).
5. Ask clarifying questions when the query is ambiguous.

Candidate Resume:
{resume_context}"""

print("All prompt templates defined.")
print(f"  - Resume Checker    : {len(RESUME_CHECKER_PROMPT.template)} chars")
print(f"  - Resume Scorer     : {len(RESUME_SCORER_PROMPT.template)} chars")
print(f"  - Cover Letter      : {len(COVER_LETTER_PROMPT.template)} chars")
print(f"  - Career Coach sys  : {len(CAREER_COACH_SYSTEM)} chars")

All prompt templates defined.
  - Resume Checker    : 1185 chars
  - Resume Scorer     : 1885 chars
  - Cover Letter      : 955 chars
  - Career Coach sys  : 861 chars


---
## Module 1 — Resume Checker
> **What it does:** Evaluates your resume standalone — no job description needed.  
> Produces a score out of 100, detailed strengths/weaknesses, skills inventory,  
> recommended skill additions, and realistic next career steps.

**How to run:**
1. Execute the cell below
2. Use the upload widget to select your resume PDF
3. Results stream to screen and are saved to MongoDB + MLflow


In [14]:
# ── MODULE 1: Resume Checker ──────────────────────────────────────────────
print("=" * 60)
print("MODULE 1: Resume Checker")
print("=" * 60)

# Step 1: Upload & extract resume PDF
# -------------------------------------------------------------------------
# Colab file upload dialog will appear when you run this cell.
# Select your resume PDF from your local machine.
resume_text_m1, filename_m1 = upload_and_extract(
    "Upload your resume PDF for evaluation:"
)

# Truncate resume text if it's too long to prevent LLM context window errors.
# The 'llama-3.1-8b-instant' model has a context window of 8192 tokens.
# We'll aim for a safe resume text length (e.g., ~4000 characters) to leave sufficient
# room for prompts, job descriptions, and generated output.
MAX_RESUME_CHARS = 4000 # Approximately 1000 tokens (assuming 1 token ~ 4 chars)
if len(resume_text_m1) > MAX_RESUME_CHARS:
    print(f"WARNING: Resume text is very long ({len(resume_text_m1)} chars). Truncating to {MAX_RESUME_CHARS} chars for LLM processing to avoid context window errors.")
    resume_text_m1 = resume_text_m1[:MAX_RESUME_CHARS]

# Step 2: Build and invoke the LLM chain
# -------------------------------------------------------------------------
print("\nEvaluating your resume... (streaming output below)")
print("-" * 60)

llm_checker = get_llm(temperature=0.1, max_tokens=2000)
chain_m1    = RESUME_CHECKER_PROMPT | llm_checker

# Stream the response token-by-token for real-time feedback
full_response_m1 = ""
t_start = time.time()

for chunk in chain_m1.stream({"context": resume_text_m1}):
    token = chunk.content if hasattr(chunk, "content") else str(chunk)
    full_response_m1 += token
    print(token, end="", flush=True)   # Stream tokens to Colab output

latency_m1 = round(time.time() - t_start, 2)

# Step 3: Extract numeric score for metrics logging
# -------------------------------------------------------------------------
score_m1 = parse_score_from_output(full_response_m1)

# Step 4: Persist to MongoDB + MLflow
# -------------------------------------------------------------------------
run_id_m1 = log_run(
    tool    = "resume_checker",
    inputs  = {"filename": filename_m1, "resume_chars": len(resume_text_m1)},
    output  = full_response_m1,
    metrics = {
        "latency_sec": latency_m1,
        "score":       score_m1 or 0.0,
        "output_chars": len(full_response_m1)
    }
)

print(f"\n\n{'=' * 60}")
print(f"Resume Checker complete.")
print(f"  Run ID    : {run_id_m1}")
print(f"  Score     : {score_m1}/100" if score_m1 else "  Score     : (see output above)")
print(f"  Latency   : {latency_m1}s")
print(f"  Logged to : MongoDB + MLflow")

MODULE 1: Resume Checker
Upload your resume PDF for evaluation:


Saving Sudipta Bera. UPDATED RESUME.pdf to Sudipta Bera. UPDATED RESUME (2).pdf


/tmp/ipykernel_5940/2251143651.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "uploaded_at": datetime.datetime.utcnow().isoformat()},


Extracted 11382 characters from 'Sudipta Bera. UPDATED RESUME (2).pdf'.
Preview (first 300 chars):
M S  O f f i c e  S u i t e
C r e a t i v e  C o n t e n t  W r i t e  - U p
A d o b e  P h o t o s h o p  ( B e g i n n e r )
C a n v a  ( I n t e r m e d i a t e )
P o w e r  B I   ( B e g i n n e r )
S Q L   ( B e g i n n e r )
D i g i t a l  M a r k e t i n g   
P r o j e c t  M a n a g e m e n ...

Evaluating your resume... (streaming output below)
------------------------------------------------------------
1. Score: 72/100
The score is based on the content, structure, and overall effectiveness of the resume. The resume has some strengths, but it also has areas for improvement, particularly in terms of clarity, concision, and relevance to the target job market.

2. Strengths:
   - The resume highlights the candidate's experience in content strategy, brand management, client servicing, and social media amplification, which are valuable skills in the communication and marketing indust

/tmp/ipykernel_5940/2883982646.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts     = datetime.datetime.utcnow().isoformat()


🏃 View run resume_checker_d896ffe4 at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0/runs/0ef6477519f84454b4749afca5259711
🧪 View experiment at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0


Resume Checker complete.
  Run ID    : d896ffe4
  Score     : 72.0/100
  Latency   : 1.71s
  Logged to : MongoDB + MLflow


---
## Module 2 — Resume Scorer / JD Matcher
> **What it does:** Deep match analysis between your resume and a specific job description.  
> Returns score, match %, keyword gaps, ATS score, readability score, skill gap analysis,  
> improvement suggestions, and industry-specific feedback.

**How to run:**
1. Paste the job description into the `job_description` variable below
2. Execute the cell — upload dialog will appear for your PDF
3. Full analysis streams to screen


In [15]:
# ── MODULE 2: Resume Scorer / JD Matcher ─────────────────────────────────
print("=" * 60)
print("MODULE 2: Resume Scorer / JD Matcher")
print("=" * 60)

# ── CONFIGURE: Paste your target job description here ─────────────────────
job_description_m2 = """
[PASTE YOUR JOB DESCRIPTION HERE]

Example (replace this entire block):
Senior Data Scientist - Machine Learning Platform
We are looking for a Senior Data Scientist to join our ML Platform team.

Responsibilities:
- Design, develop, and deploy scalable ML models in production
- Lead end-to-end model lifecycle: data collection, feature engineering, training, evaluation, deployment
- Collaborate with engineering teams on MLOps practices (CI/CD, monitoring, drift detection)
- Mentor junior data scientists

Requirements:
- 5+ years experience in applied machine learning
- Strong proficiency in Python, TensorFlow or PyTorch
- Experience with cloud platforms (AWS, GCP, or Azure)
- Familiarity with MLflow, Kubeflow, or similar MLOps tooling
- MS or PhD in Computer Science, Statistics, or related field preferred
"""

# Step 1: Upload resume PDF
# -------------------------------------------------------------------------
# Re-use already-extracted text if Module 1 was run (saves re-upload)
if 'resume_text_m1' in dir() and resume_text_m1:
    resume_text_m2 = resume_text_m1
    filename_m2    = filename_m1
    print(f"Reusing resume from Module 1: '{filename_m1}'")
else:
    resume_text_m2, filename_m2 = upload_and_extract(
        "Upload your resume PDF for JD matching:"
    )

# Validate job description was actually filled in
if "[PASTE YOUR JOB DESCRIPTION HERE]" in job_description_m2:
    print("\nWARNING: Using the example JD above. Replace with your real job description for accurate results.")

# Step 2: Build and invoke scorer chain
# -------------------------------------------------------------------------
print("\nScoring resume against job description... (streaming)")
print("-" * 60)

llm_scorer = get_llm(temperature=0.2, max_tokens=2200)
chain_m2   = RESUME_SCORER_PROMPT | llm_scorer

full_response_m2 = ""
t_start = time.time()

for chunk in chain_m2.stream({
    "job_description": job_description_m2.strip(),
    "context":         resume_text_m2
}):
    token = chunk.content if hasattr(chunk, "content") else str(chunk)
    full_response_m2 += token
    print(token, end="", flush=True)

latency_m2 = round(time.time() - t_start, 2)

# Step 3: Parse scores from output for metrics
# -------------------------------------------------------------------------
score_m2 = parse_score_from_output(full_response_m2)

# Extract ATS score if present
ats_match = re.search(r'ATS Compatibility Score[:\s]+([0-9]+)', full_response_m2)
ats_score = float(ats_match.group(1)) if ats_match else 0.0

# Extract overall match %
match_pct_match = re.search(r'Overall Match[:\s]+([0-9]+)%', full_response_m2)
match_pct = float(match_pct_match.group(1)) if match_pct_match else 0.0

# Step 4: Persist results
# -------------------------------------------------------------------------
run_id_m2 = log_run(
    tool    = "resume_scorer",
    inputs  = {"filename": filename_m2, "jd_chars": len(job_description_m2)},
    output  = full_response_m2,
    metrics = {
        "latency_sec":    latency_m2,
        "score":          score_m2 or 0.0,
        "ats_score":      ats_score,
        "match_pct":      match_pct,
        "output_chars":   len(full_response_m2)
    }
)

print(f"\n\n{'=' * 60}")
print(f"Resume Scorer complete.")
print(f"  Run ID      : {run_id_m2}")
print(f"  Score       : {score_m2}/100" if score_m2 else "  Score       : (see output)")
print(f"  ATS Score   : {int(ats_score)}/100")
print(f"  Match %     : {int(match_pct)}%")
print(f"  Latency     : {latency_m2}s")
print(f"  Logged to   : MongoDB + MLflow")

MODULE 2: Resume Scorer / JD Matcher
Reusing resume from Module 1: 'Sudipta Bera. UPDATED RESUME (2).pdf'


Scoring resume against job description... (streaming)
------------------------------------------------------------
**Score: 24/100**
**Overall Match: 12%**

**Keywords matched:**
- Social Media Management
- Content Management System
- Digital Marketing
- Event Coordination
- Client Servicing
- Marketing Research
- Corporate Communications

**Missing keywords:**
- Machine Learning
- Applied Machine Learning
- Python
- TensorFlow or PyTorch
- Cloud Platforms (AWS, GCP, or Azure)
- MLOps tooling (MLflow, Kubeflow)
- Data Collection
- Feature Engineering
- Model Training
- Model Deployment
- Model Evaluation
- Drift Detection
- CI/CD
- Monitoring

**Readability Score: 60/100**
The resume is written in a clear and concise manner, but there are some formatting issues and excessive use of all caps for job titles. The language is mostly professional, but there are some minor grammatical 

/tmp/ipykernel_5940/2883982646.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts     = datetime.datetime.utcnow().isoformat()


🏃 View run resume_scorer_0f12de8d at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0/runs/9b97e045ca6d4fdbaeaae41f349c9bfe
🧪 View experiment at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0


Resume Scorer complete.
  Run ID      : 0f12de8d
  Score       : 24.0/100
  ATS Score   : 40/100
  Match %     : 12%
  Latency     : 34.69s
  Logged to   : MongoDB + MLflow


---
## Module 3 — Cover Letter Generator
> **What it does:** Generates a tailored, professional cover letter (300-450 words)  
> by combining your resume content with the target job description.  
> Output is streamed in real-time and saved as a `.txt` file.

**How to run:**
1. Fill in `job_description_m3` below with the target JD
2. Optionally set `your_name` for personalisation
3. Execute — cover letter streams to screen and saves locally


In [16]:
# ── MODULE 3: Cover Letter Generator ─────────────────────────────────────
print("=" * 60)
print("MODULE 3: Cover Letter Generator")
print("=" * 60)

# ── CONFIGURE: Paste your target job description ──────────────────────────
job_description_m3 = """
[PASTE YOUR JOB DESCRIPTION HERE]

Example:
Machine Learning Engineer - AI Products Team
We're seeking an ML Engineer to build and ship ML-powered product features.
Requirements: Python, PyTorch/TensorFlow, MLOps, cloud deployment (AWS/GCP).
Nice to have: LLM fine-tuning, RAG systems, Kubernetes.
"""

# Step 1: Resume - reuse or re-upload
# -------------------------------------------------------------------------
if 'resume_text_m1' in dir() and resume_text_m1:
    resume_text_m3 = resume_text_m1
    filename_m3    = filename_m1
    print(f"Reusing resume: '{filename_m1}'")
else:
    resume_text_m3, filename_m3 = upload_and_extract(
        "Upload your resume PDF for cover letter generation:"
    )

# Step 2: Generate cover letter (streaming)
# -------------------------------------------------------------------------
print("\nGenerating tailored cover letter... (streaming)")
print("-" * 60)

# Slightly higher temperature for creative, natural writing tone
llm_cl    = get_llm(temperature=0.3, max_tokens=1500)
chain_m3  = COVER_LETTER_PROMPT | llm_cl

cover_letter_text = ""
t_start = time.time()

for chunk in chain_m3.stream({
    "job_description": job_description_m3.strip(),
    "resume_text":     resume_text_m3
}):
    token = chunk.content if hasattr(chunk, "content") else str(chunk)
    cover_letter_text += token
    print(token, end="", flush=True)

latency_m3 = round(time.time() - t_start, 2)

# Step 3: Save cover letter to file (downloadable from Colab)
# -------------------------------------------------------------------------
output_filename = f"Cover_Letter_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(cover_letter_text)

# Step 4: Persist to MongoDB + MLflow
# -------------------------------------------------------------------------
run_id_m3 = log_run(
    tool    = "cover_letter_generator",
    inputs  = {"filename": filename_m3, "jd_chars": len(job_description_m3)},
    output  = cover_letter_text,
    metrics = {
        "latency_sec":    latency_m3,
        "output_words":   len(cover_letter_text.split()),
        "output_chars":   len(cover_letter_text)
    }
)

print(f"\n\n{'=' * 60}")
print(f"Cover Letter generated.")
print(f"  Run ID        : {run_id_m3}")
print(f"  Word count    : {len(cover_letter_text.split())} words")
print(f"  Latency       : {latency_m3}s")
print(f"  Saved to      : {output_filename}")
print(f"  Logged to     : MongoDB + MLflow")

# Download the file directly from Colab
from google.colab import files
files.download(output_filename)
print(f"  Download triggered for: {output_filename}")

MODULE 3: Cover Letter Generator
Reusing resume: 'Sudipta Bera. UPDATED RESUME (2).pdf'

Generating tailored cover letter... (streaming)
------------------------------------------------------------
[Your Name]
[Your Address]
[City, State, Zip]
[Email Address]
[Phone Number]
[Date]

[Recipient’s Name]
[Recipient’s Title]
[Company Name]
[Company Address]
[City, State, Zip]

Dear [Recipient’s Name],

I am thrilled to apply for the Machine Learning Engineer position at [Company Name], where I can leverage my technical skills and passion for AI to build and ship ML-powered product features. With a strong foundation in public relations and event management, I am confident that my unique blend of skills will enable me to make a significant impact in this role.

As a creative professional with hands-on experience in content strategy, brand management, and social media amplification, I have developed a keen understanding of audience engagement and digital marketing. My experience as a Content W

/tmp/ipykernel_5940/2883982646.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts     = datetime.datetime.utcnow().isoformat()


🏃 View run cover_letter_generator_b4bebfa3 at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0/runs/231a6d4e29db45aaace64735c5a3f124
🧪 View experiment at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0


Cover Letter generated.
  Run ID        : b4bebfa3
  Word count    : 305 words
  Latency       : 34.26s
  Saved to      : Cover_Letter_20260410_152919.txt
  Logged to     : MongoDB + MLflow


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Download triggered for: Cover_Letter_20260410_152919.txt


---
## Module 4 — AI Career Coach (Multi-Turn Chatbot)
> **What it does:** A multi-turn conversational career coach grounded in your resume.  
> Ask about career paths, interview prep, skill gaps, salary negotiation, LinkedIn optimization.  
> Full conversation history is persisted to MongoDB.

**How to run:**
1. Execute the cell — upload dialog appears for your PDF
2. Type your questions at the `You:` prompt
3. Type `quit` or `exit` to end the session
4. Full conversation is saved to MongoDB automatically


In [17]:
# ── MODULE 4: AI Career Coach ─────────────────────────────────────────────
print("=" * 60)
print("MODULE 4: AI Career Coach (Multi-Turn Chat)")
print("=" * 60)

# Step 1: Upload resume — grounds the coach in your actual experience
# -------------------------------------------------------------------------
if 'resume_text_m1' in dir() and resume_text_m1:
    resume_text_m4 = resume_text_m1
    filename_m4    = filename_m1
    print(f"Reusing resume: '{filename_m1}'")
else:
    resume_text_m4, filename_m4 = upload_and_extract(
        "Upload your resume to start Career Coach session:"
    )

# Step 2: Initialise LLM and session state
# -------------------------------------------------------------------------
llm_coach    = get_llm(temperature=0.4, max_tokens=1000)
chat_history = []      # Accumulates HumanMessage / AIMessage objects
session_id   = str(uuid.uuid4())[:8]

# Build the grounded system message from the candidate's resume
system_message = SystemMessage(
    content=CAREER_COACH_SYSTEM.format(resume_context=resume_text_m4)
)

print(f"\nCareer Coach session started. (Session ID: {session_id})")
print(f"Resume loaded: '{filename_m4}'")
print("\nExample questions to ask:")
print("  - What roles am I best suited for based on my resume?")
print("  - How should I prepare for a data science interview?")
print("  - What skills should I add to become more competitive?")
print("  - How do I explain my career gap in an interview?")
print("  - Critique my resume's summary section.")
print("\nType 'quit' or 'exit' to end the session.")
print("-" * 60)

# Step 3: Multi-turn conversation loop
# -------------------------------------------------------------------------
turn = 0
while True:
    # Get user input (blocking call - works in Colab)
    try:
        user_input = input("\nYou: ").strip()
    except EOFError:
        # Handles non-interactive execution gracefully
        print("\n[Non-interactive mode detected - ending session]")
        break

    if not user_input:
        print("(Empty input — please type a question or 'quit' to exit)")
        continue

    if user_input.lower() in ['quit', 'exit', 'bye', 'done']:
        print("\nCareer Coach: Goodbye! Best of luck with your job search!")
        break

    # Append user message to history
    chat_history.append(HumanMessage(content=user_input))
    turn += 1

    # Build full message list: [system] + [all history so far]
    messages = [system_message] + chat_history

    # Stream assistant response
    print("\nCareer Coach: ", end="", flush=True)
    response_text = ""
    t_start = time.time()

    for chunk in llm_coach.stream(messages):
        token = chunk.content if hasattr(chunk, "content") else str(chunk)
        response_text += token
        print(token, end="", flush=True)

    latency_turn = round(time.time() - t_start, 2)
    print()    # Newline after streamed response

    # Append assistant response to history
    chat_history.append(AIMessage(content=response_text))

    # Log each turn to MongoDB for full session replay capability
    col_chat_sessions.insert_one({
        "session_id":  session_id,
        "turn":        turn,
        "timestamp":   datetime.datetime.utcnow().isoformat(),
        "filename":    filename_m4,
        "user":        user_input,
        "assistant":   response_text,
        "latency_sec": latency_turn
    })

# Step 4: Save full session summary to MongoDB + MLflow
# -------------------------------------------------------------------------
if chat_history:
    # Format full conversation for MLflow artifact
    convo_text = ""
    for msg in chat_history:
        role = "USER" if isinstance(msg, HumanMessage) else "COACH"
        convo_text += f"[{role}]\n{msg.content}\n\n"

    run_id_m4 = log_run(
        tool    = "career_coach",
        inputs  = {"filename": filename_m4, "session_id": session_id, "turns": turn},
        output  = convo_text,
        metrics = {"turns": float(turn), "session_chars": float(len(convo_text))}
    )

    print(f"\n{'=' * 60}")
    print(f"Career Coach session ended.")
    print(f"  Session ID   : {session_id}")
    print(f"  Turns        : {turn}")
    print(f"  Run ID       : {run_id_m4}")
    print(f"  Logged to    : MongoDB + MLflow")
else:
    print("\nNo messages recorded — session ended without interaction.")

MODULE 4: AI Career Coach (Multi-Turn Chat)
Reusing resume: 'Sudipta Bera. UPDATED RESUME (2).pdf'

Career Coach session started. (Session ID: 77ed7695)
Resume loaded: 'Sudipta Bera. UPDATED RESUME (2).pdf'

Example questions to ask:
  - What roles am I best suited for based on my resume?
  - How should I prepare for a data science interview?
  - What skills should I add to become more competitive?
  - How do I explain my career gap in an interview?
  - Critique my resume's summary section.

Type 'quit' or 'exit' to end the session.
------------------------------------------------------------

You: What roles am I best suited for based on my resume?

Career Coach: Based on your resume, you have a versatile background in creative content writing, digital marketing, project management, and social media management. Here are some roles you may be well-suited for:

1. **Content Marketing Specialist**: Your experience in content writing, content management systems, and niche marketing of the

/tmp/ipykernel_5940/3926167954.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":   datetime.datetime.utcnow().isoformat(),



You: quit

Career Coach: Goodbye! Best of luck with your job search!


/tmp/ipykernel_5940/2883982646.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts     = datetime.datetime.utcnow().isoformat()


🏃 View run career_coach_014108d7 at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0/runs/879d7528352041d192d38d31e726aefe
🧪 View experiment at: https://dagshub.com/prithusarkar90/resume-genie.mlflow/#/experiments/0

Career Coach session ended.
  Session ID   : 77ed7695
  Turns        : 1
  Run ID       : 014108d7
  Logged to    : MongoDB + MLflow


---
## Full Pipeline Demo — All 4 Modules on a Single Resume
> Run this cell to execute **all four modules sequentially** on one PDF upload.  
> Useful for a quick end-to-end demonstration. Uses a synthetic demo JD so you can  
> see all output shapes without needing a real job description.


In [20]:
# ── FULL PIPELINE DEMO ────────────────────────────────────────────────────
print("=" * 60)
print("FULL PIPELINE DEMO — Resume Genie Suite")
print("=" * 60)

# Demo job description used across Modules 2 + 3
DEMO_JD = """
Senior AI/ML Engineer
We are building the next generation AI platform and are looking for a
Senior AI/ML Engineer with strong Python skills, experience in LLMs,
RAG systems, LangChain, and MLOps practices (MLflow, DVC).
You will lead model development, deployment, and monitoring pipelines.
Requirements: 5+ years Python, LLM experience, cloud (AWS/GCP), MLOps tooling.
Nice to have: Fine-tuning, vector databases, real-time inference systems.
"""

# Upload once — reused across all modules
print("\nStep 0: Upload your resume PDF (used for all 4 modules)")
demo_resume_text, demo_filename = upload_and_extract(
    "Upload your resume for the full pipeline demo:"
)

# Truncate resume text to prevent LLM context window errors, using the same MAX_RESUME_CHARS
# defined in Module 1 (currently 4000 chars)
if len(demo_resume_text) > MAX_RESUME_CHARS:
    print(f"WARNING: Resume text is very long ({len(demo_resume_text)} chars). Truncating to {MAX_RESUME_CHARS} chars for LLM processing to avoid context window errors.")
    demo_resume_text = demo_resume_text[:MAX_RESUME_CHARS]

demo_results = {}
pipeline_start = time.time()

# ── Module 1: Resume Checker ──────────────────────────────────────────────
print("\n" + "=" * 40)
print("DEMO — Module 1: Resume Checker")
print("=" * 40)
llm_d1   = get_llm(temperature=0.1, max_tokens=2000)
t0 = time.time()
resp_d1  = (RESUME_CHECKER_PROMPT | llm_d1).invoke({"context": demo_resume_text})
out_d1   = resp_d1.content
lat_d1   = round(time.time() - t0, 2)
score_d1 = parse_score_from_output(out_d1)
print(out_d1[:1500] + ("..." if len(out_d1) > 1500 else ""))
demo_results['checker'] = {"score": score_d1, "latency": lat_d1, "output_len": len(out_d1)}

# ── Module 2: Resume Scorer ───────────────────────────────────────────────
print("\n" + "=" * 40)
print("DEMO — Module 2: Resume Scorer vs JD")
print("=" * 40)
llm_d2  = get_llm(temperature=0.2, max_tokens=2200)
t0 = time.time()
resp_d2 = (RESUME_SCORER_PROMPT | llm_d2).invoke(
    {
    "job_description": DEMO_JD.strip(),
    "context": demo_resume_text
    }
)
out_d2  = resp_d2.content
lat_d2  = round(time.time() - t0, 2)
score_d2= parse_score_from_output(out_d2)
print(out_d2[:1500] + ("..." if len(out_d2) > 1500 else ""))
demo_results['scorer'] = {"score": score_d2, "latency": lat_d2}

# ── Module 3: Cover Letter ────────────────────────────────────────────────
print("\n" + "=" * 40)
print("DEMO — Module 3: Cover Letter Generator")
print("=" * 40)
llm_d3  = get_llm(temperature=0.3, max_tokens=1500)
t0 = time.time()
resp_d3 = (COVER_LETTER_PROMPT | llm_d3).invoke(
    {
    "job_description": DEMO_JD.strip(),
    "resume_text": demo_resume_text
    }
)
out_d3  = resp_d3.content
lat_d3  = round(time.time() - t0, 2)
print(out_d3)
demo_results['cover_letter'] = {"words": len(out_d3.split()), "latency": lat_d3}

# Save cover letter to file
cl_file = f"Cover_Letter_Demo_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(cl_file, "w") as f:
    f.write(out_d3)

# ── Module 4: Career Coach (single turn demo) ─────────────────────────────
print("\n" + "=" * 40)
print("DEMO — Module 4: Career Coach (sample turn)")
print("=" * 40)
llm_d4     = get_llm(temperature=0.4, max_tokens=800)
sys_demo   = SystemMessage(content=CAREER_COACH_SYSTEM.format(resume_context=demo_resume_text))
demo_q     = HumanMessage(content="Based on my resume, what are the top 3 improvements I should make to land a senior AI/ML engineering role?")
t0 = time.time()
resp_d4    = llm_d4.invoke([sys_demo, demo_q])
out_d4     = resp_d4.content
lat_d4     = round(time.time() - t0, 2)
print(f"Q: {demo_q.content}")
print(f"\nCareer Coach:\n{out_d4}")
demo_results['career_coach'] = {"latency": lat_d4, "output_len": len(out_d4)}

# ── Log full pipeline run to MLflow ───────────────────────────────────────
pipeline_total = round(time.time() - pipeline_start, 2)
with mlflow.start_run(run_name=f"full_pipeline_demo_{str(uuid.uuid4())[:6]}"):
    mlflow.log_param("filename",        demo_filename)
    mlflow.log_param("llm",             LLM_PROVIDER)
    mlflow.log_param("model",           GROQ_MODEL if LLM_PROVIDER == "groq" else OPENAI_MODEL)
    mlflow.log_metric("total_latency",  pipeline_total)
    mlflow.log_metric("checker_score",  score_d1 or 0.0)
    mlflow.log_metric("scorer_score",   score_d2 or 0.0)
    mlflow.log_metric("cover_letter_words", len(out_d3.split()))
    mlflow.log_text(out_d1, "module1_checker.txt")
    mlflow.log_text(out_d2, "module2_scorer.txt")
    mlflow.log_text(out_d3, "module3_cover_letter.txt")
    mlflow.log_text(out_d4, "module4_coach.txt")

# ── Log to MongoDB ─────────────────────────────────────────────────────────
col_runs.insert_one({
    "run_id":         f"demo_{str(uuid.uuid4())[:8]}",
    "tool":           "full_pipeline_demo",
    "timestamp":      datetime.datetime.utcnow().isoformat(),
    "filename":       demo_filename,
    "total_latency":  pipeline_total,
    "results":        demo_results
})

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FULL PIPELINE DEMO COMPLETE")
print("=" * 60)
print(f"  Resume Checker  — Score  : {score_d1}/100  | Latency: {lat_d1}s")
print(f"  Resume Scorer   — Score  : {score_d2}/100  | Latency: {lat_d2}s")
print(f"  Cover Letter    — Words  : {len(out_d3.split())}       | Latency: {lat_d3}s")
print(f"  Career Coach    — Demo Q | Latency: {lat_d4}s")
print(f"  Total Pipeline  — Latency: {pipeline_total}s")
print(f"  Logged to       — MongoDB + MLflow")
print(f"  Cover Letter    — Saved: {cl_file}")

# Download cover letter
from google.colab import files
files.download(cl_file)

FULL PIPELINE DEMO — Resume Genie Suite

Step 0: Upload your resume PDF (used for all 4 modules)
Upload your resume for the full pipeline demo:


Saving Sudipta Bera. UPDATED RESUME.pdf to Sudipta Bera. UPDATED RESUME (5).pdf


/tmp/ipykernel_5940/2251143651.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "uploaded_at": datetime.datetime.utcnow().isoformat()},


Extracted 11382 characters from 'Sudipta Bera. UPDATED RESUME (5).pdf'.
Preview (first 300 chars):
M S  O f f i c e  S u i t e
C r e a t i v e  C o n t e n t  W r i t e  - U p
A d o b e  P h o t o s h o p  ( B e g i n n e r )
C a n v a  ( I n t e r m e d i a t e )
P o w e r  B I   ( B e g i n n e r )
S Q L   ( B e g i n n e r )
D i g i t a l  M a r k e t i n g   
P r o j e c t  M a n a g e m e n ...

DEMO — Module 1: Resume Checker
1. Score: 72/100
The score is based on the content, structure, and overall effectiveness of the resume. The resume has some strengths, but it also has areas for improvement, particularly in terms of clarity, concision, and relevance to the target job market.

2. Strengths:
   - The resume highlights the candidate's experience in content creation, digital marketing, and project management, which are valuable skills in today's job market.
   - The candidate has a strong foundation in public relations and event management, as evident from their Master's degree 

/tmp/ipykernel_5940/783360049.py:119: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":      datetime.datetime.utcnow().isoformat(),



FULL PIPELINE DEMO COMPLETE
  Resume Checker  — Score  : 72.0/100  | Latency: 1.33s
  Resume Scorer   — Score  : 22.0/100  | Latency: 38.81s
  Cover Letter    — Words  : 327       | Latency: 17.03s
  Career Coach    — Demo Q | Latency: 28.12s
  Total Pipeline  — Latency: 85.93s
  Logged to       — MongoDB + MLflow
  Cover Letter    — Saved: Cover_Letter_Demo_20260410_153245.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Analytics — MLflow Runs & MongoDB Query
> View all logged runs, compare scores across invocations, and inspect MongoDB records.


In [21]:
# ── MLflow Experiment Summary ─────────────────────────────────────────────
print("MLflow Experiment: resume_genie_suite")
print("-" * 60)

runs = mlflow.search_runs(experiment_names=["resume_genie_suite"])

if not runs.empty:
    # Select most informative columns for display
    display_cols = [c for c in [
        "run_id", "params.tool", "params.model",
        "metrics.score", "metrics.latency_sec",
        "metrics.match_pct", "metrics.ats_score",
        "start_time"
    ] if c in runs.columns]
    print(runs[display_cols].to_string(index=False))
else:
    print("No runs logged yet. Execute the modules above first.")

# ── MongoDB Recent Runs ───────────────────────────────────────────────────
print("\n" + "-" * 60)
print("MongoDB: Last 5 tool runs")
print("-" * 60)

for doc in col_runs.find().sort("timestamp", -1).limit(5):
    print(f"  [{doc.get('timestamp','')}]  tool={doc.get('tool','?'):<25}  "
          f"run_id={doc.get('run_id','?')}")

print("\n" + "-" * 60)
print("MongoDB: Last 3 chat session turns")
print("-" * 60)

for turn in col_chat_sessions.find().sort("timestamp", -1).limit(3):
    print(f"  [{turn.get('timestamp','')}] session={turn.get('session_id','?')}  "
          f"turn={turn.get('turn','?')}")
    print(f"    USER : {turn.get('user','')[:80]}")
    print(f"    COACH: {turn.get('assistant','')[:80]}...")
    print()

MLflow Experiment: resume_genie_suite
------------------------------------------------------------
                          run_id            params.tool         params.model  metrics.score  metrics.latency_sec  metrics.match_pct  metrics.ats_score                       start_time
fcfd6def5ae14ba384cbcb37f94cc2b7                   None llama-3.1-8b-instant            NaN                  NaN                NaN                NaN 2026-04-10 15:33:13.804000+00:00
879d7528352041d192d38d31e726aefe           career_coach llama-3.1-8b-instant            NaN                  NaN                NaN                NaN 2026-04-10 15:29:46.546000+00:00
231a6d4e29db45aaace64735c5a3f124 cover_letter_generator llama-3.1-8b-instant            NaN                34.26                NaN                NaN 2026-04-10 15:29:20.122000+00:00
9b97e045ca6d4fdbaeaae41f349c9bfe          resume_scorer llama-3.1-8b-instant           24.0                34.69               12.0               40.0 2026-04-10 15: